# JW $Z_4^{TF}$ system - cocycle extraction - larger system

Created: 16-09-2026

Objectives:
* Iterate on [this notebook](jw_z_4_tf_system_cocycle_extraction.ipynb), apply to systems with random circuits.

# Imports

In [1]:
import numpy as np

In [2]:
import jax
jax.config.update('jax_platform_name', 'cpu')

import jax.numpy as jnp

In [3]:
import matplotlib.pyplot as plt

In [4]:
from tqdm import tqdm

In [5]:
from functools import reduce
from operator import mul
from itertools import product

In [6]:
from random import random

In [7]:
import quimb.tensor as qtn
import quimb as qu

In [8]:
from scipy.stats import unitary_group

In [9]:
from collections import Counter

In [10]:
import pandas as pd

In [11]:
from time import time

In [12]:
from humanize import naturalsize

In [108]:
import pickle

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


# Definitions
## Construct cluster state

In [13]:
np_up_X_state = 1/(np.sqrt(2))*np.array([1,1])

In [14]:
qu_up_X_state = qtn.Tensor(
    data=np_up_X_state,
    inds=('k',),
    tags='prod'
)

In [15]:
np_CZ = np.diag([1,1,1,-1])

In [16]:
np_CZ = np_CZ.reshape((2,)*4)

In [17]:
qu_CZ = qtn.Tensor(
    data=np_CZ,
    inds=('k1', 'k2', 'b1', 'b2'),
    tags='CZ'
)

In [18]:
np_hadamard = np.pow(2, -1/2)*np.array([
    [1,1],
    [1,-1]
])

In [19]:
qu_hadamard = qtn.Tensor(data=np_hadamard, inds=('k', 'b'), tags='Had')

In [20]:
def get_cluster_state_qu_tensor_network(num_sites):
    assert (num_sites%2) == 0

    product_state_tensors = [
        qu_up_X_state.reindex({'k': f'kc_1_{i}'})
        for i in range(num_sites)
    ]

    first_layer_circuit_tensors = [
        qu_CZ.reindex({
            'b1': f'kc_1_{i}',
            'b2': f'kc_1_{i+1}',
            'k1': f'kc_2_{i}',
            'k2': f'kc_2_{i+1}'
        })
        for i in range(0, num_sites, 2)
    ]


    second_layer_circuit_tensors = [
        qu_CZ.reindex({
            'b1': f'kc_2_{i}',
            'b2': f'kc_2_{(i+1)%num_sites}',
            'k1': f'kh_{i}',
            'k2': f'k{(i+1)%num_sites}'
        })
        for i in range(1, num_sites+1, 2)
    ]

    hadamard_layer = [
        qu_hadamard.reindex({
            'b': f'kh_{i}',
            'k': f'k{i}'
        })
        for i in range(1, num_sites, 2)
    ]
    all_tensors = (
        product_state_tensors
        + first_layer_circuit_tensors
        + second_layer_circuit_tensors
        + hadamard_layer
    )

    out = qtn.TensorNetwork(all_tensors, virtual=True)
    out.mangle_inner_()

    return out

## Construct product state

In [21]:
np_up_X_state = 1/(np.sqrt(2))*np.array([1,1])

In [22]:
np_up_Z_state = np.array([1,0])

In [23]:
qu_up_Z_state = qtn.Tensor(
    data=np_up_Z_state,
    inds=('k',),
    tags='prod'
)

In [24]:
alternating_states = [
    qu_up_X_state,
    qu_up_Z_state
]

def get_product_qu_tensor_network(num_sites):
    assert (num_sites%2) == 0

    product_state_tensors = [
        alternating_states[i%2].reindex({'k': f'k{i}'})
        for i in range(num_sites)
    ]

    out = qtn.TensorNetwork(
        product_state_tensors,
        virtual=True
    )
    out.mangle_inner_()

    return out

## Symmetries

In [25]:
def multikron(arrays):
    return reduce(np.kron, arrays)

In [26]:
np_I = np.array([
    [1,0],
    [0,1]
])

np_X = np.array([
    [0,1],
    [1,0]
])

np_Y = np.array([
    [0,-1j],
    [1j,0]
])

np_Z = np.array([
    [1,0],
    [0,-1]
])

In [27]:
qu_I = qtn.Tensor(
    np_I,
    inds=['k', 'b'],
    tags='X'
)

qu_X = qtn.Tensor(
    np_X,
    inds=['k', 'b'],
    tags='X'
)

qu_Y = qtn.Tensor(
    np_Y,
    inds=['k', 'b'],
    tags='Y'
)

qu_Z = qtn.Tensor(
    np_Z,
    inds=['k', 'b'],
    tags='Z'
)

In [28]:
def get_multisite_qu_X(num_sites):
    np_many_X = multikron([np_X]*num_sites)

    out = qtn.Tensor(
        np_many_X,
        inds=['k', 'b'],
        tags='mulit_site_X',
    )

    return out

In [29]:
def get_multisite_qu_I(num_sites):
    np_many_I = multikron([np_I]*num_sites)

    out = qtn.Tensor(
        np_many_I,
        inds=['k', 'b'],
        tags='mulit_site_I',
    )

    return out

In [30]:
qu_spin_fermion_fp = (
    qu_I.reindex({'k': 'ks', 'b': 'bs'})
    & qu_Z.reindex({'k': 'kf', 'b': 'bf'})
).contract()

qu_unit_cell_fp = qu_spin_fermion_fp.fuse({
    'k': ['ks', 'kf'],
    'b': ['bs', 'bf']
})

In [31]:
"""
def get_multisite_qu_fp(num_sites):
    assert (num_sites%2)==0

    num_unit_cells = (num_sites//2)

    np
"""

'\ndef get_multisite_qu_fp(num_sites):\n    assert (num_sites%2)==0\n\n    num_unit_cells = (num_sites//2)\n\n    np\n'

Decompose T symmetry as $MK$:

In [32]:
np_00 = np.array([[1,0], [0,0]])
np_11 = np.array([[0,0], [0,1]])

In [33]:
def tensor_product_operators(op_1, op_2):
    out = (
        op_1[..., np.newaxis, np.newaxis]
        *op_2[np.newaxis, np.newaxis, ...]
    )

    return out

In [34]:
np_M = (
    tensor_product_operators(np_X, np_00)
    + tensor_product_operators(np_Y, np_11)
)

In [35]:
qu_M = qtn.Tensor(
    np_M,
    inds=['ks', 'bs', 'kf', 'bf']
)

In [36]:
np_M_reindexed = (
    qu_M
    .fuse({
        'k': ['ks', 'kf'],
        'b': ['bs', 'bf']
    })
    .transpose('k', 'b')
    .data
)

## Extracting projectors

In [37]:
def random_uniform_complex(shape):
    return np.random.uniform(size=shape) + 1j*np.random.uniform(size=shape)

In [38]:
def maximize_projector_states(rho, left_sites, proj_sites, right_sites):
    v = qtn.Tensor(
        data=random_uniform_complex((2,)*(len(proj_sites)-1)),
        inds=[f'k{i}' for i in proj_sites[:-1]]
    )

    tnopt = qtn.TNOptimizer(
        v,  # the tensor network we want to optimize
        loss_func,  # the function we want to minimize
        norm_fn=normalize_v,
        loss_constants={"rho": rho},
        loss_kwargs={
            "left_sites": left_sites,
            "proj_sites": proj_sites,
            "right_sites": right_sites,
        },
        autodiff_backend="jax",
        optimizer="L-BFGS-B",
        progbar=False
    )

    v_opt = tnopt.optimize(n=2000)

    embedded_v_opt = embed_fp_even_vector(v_opt).contract()

    return v_opt, embedded_v_opt, tnopt.losses

In [39]:
def projector_state_check(state, rho, left_sites):
    left_right_rho = (
        rho
        & state
        & state.conj().reindex({s: f'b{s[1:]}' for s in state.inds})
    )

    left_right_rho = left_right_rho.contract()

    left_inds = [
        f'{s}{i}'
        for i in left_sites
        for s in 'kb'
    ]

    schmidt_decomp = qtn.tensor_core.tensor_split(
        left_right_rho,
        left_inds=left_inds,
        method='svd',
        #cutoff=1e-6,
        cutoff_mode='abs',
        absorb=None,
        renorm=False,
        bond_ind='v'
    )

    schmidt_vals = schmidt_decomp.tensors[1]

    return schmidt_vals

### Embed vector

In [40]:
np_CX = (
    tensor_product_operators(np_00, np_I)
    + tensor_product_operators(np_11, np_X)
)

In [41]:
qu_CX = qtn.Tensor(
    np_CX,
    inds=['k1', 'b1', 'k2', 'b2']
)

In [42]:
np_up_Z_state = np.array([1,0])

In [43]:
qu_up_Z_state = qtn.Tensor(
    data=np_up_Z_state,
    inds=('k',),
    tags='Z0_pad'
)

In [44]:
def embed_fp_even_vector(v):
    # Take a vector of length 2N-1, and return a vector of length 2N
    # which commutes with IZIZ...IZIZ
    # Assuming v site ordering is spin-fermion-spin-...-fermion.

    sites = sorted(int(s[1:]) for s in v.inds)
    padded_site = sites[-1] + 1
    num_cx_gates = len(sites)//2
    
    if num_cx_gates>=1:
        padded_v = (
            v
            & qu_up_Z_state.reindex({'k': f'k{padded_site}_0'})
        )
    else:
        padded_v = (
            v
            & qu_up_Z_state.reindex({'k': f'k{padded_site}'})
        )
    num_cx_gates = len(sites)//2
    
    cx_gates = [
        qu_CX.reindex({
            'k1': f'k{sites[2*i+1]}',
            'b1': f'b{sites[2*i+1]}',
            'k2': f'k{padded_site}_{i+1}',
            'b2': f'k{padded_site}_{i}'
        })
        for i in range(num_cx_gates-1)
    ]
    
    # Handling annoying edge case logic
    if num_cx_gates >= 1:
        i = num_cx_gates-1
        cx_gates.append(
            qu_CX.reindex({
                'k1': f'k{sites[2*i+1]}',
                'b1': f'b{sites[2*i+1]}',
                'k2': f'k{padded_site}',
                'b2': f'k{padded_site}_{i}'
            })
        )
    
    reindexed_padded_v = (
        padded_v
        .reindex({
            f'k{sites[2*i+1]}': f'b{sites[2*i+1]}'
            for i in range(num_cx_gates)
        })
    )
    sym_v = qtn.TensorNetwork([
        reindexed_padded_v,
        *cx_gates
    ])
    
    sym_v.mangle_inner_()

    return sym_v

### Loss function

In [45]:
def get_rho_purity(rho, sites):
    # Assuming rho is a Hermitian reduced density matrix
    reindex_map = (
        {f'k{i}': f'b{i}' for i in sites}
        | {f'b{i}': f'k{i}' for i in sites}
    )

    rho_other = rho.reindex(reindex_map)
    #rho_other.mangle_inner_()
    
    out = (rho & rho_other).contract()

    return out

In [46]:
def normalize_v(v):
    norm = (v & v.conj()).contract()
    w = v*jnp.power(norm, -0.5)
    return w

In [47]:
def loss_func(v, rho, left_sites, proj_sites, right_sites):
    embed_v = embed_fp_even_vector(v)
    
    rho_lr = (
        rho
        & embed_v.reindex({f'k{i}': f'b{i}' for i in proj_sites})
        & embed_v.conj()
    )
    rho_lr = rho_lr.contract()
    tr_rho_lr = (
        rho_lr
        .reindex({f'k{i}': f'b{i}' for i in left_sites + right_sites})
        .contract()
    )
    
    rho_l = rho_lr.reindex(
        {f'k{i}': f'b{i}' for i in right_sites}
    )
    rho_l = rho_l.contract()*jnp.power(tr_rho_lr, -0.5)
    
    rho_r = rho_lr.reindex(
        {f'k{i}': f'b{i}' for i in left_sites}
    )
    rho_r = rho_r.contract()*jnp.power(tr_rho_lr, -0.5)
    
    purity_lr = get_rho_purity(rho_lr, left_sites + right_sites)
    purity_l = get_rho_purity(rho_l, left_sites)
    purity_r = get_rho_purity(rho_r, right_sites)
    
    reindex_map = (
        {f'k{i}': f'b{i}' for i in left_sites+right_sites}
        | {f'b{i}': f'k{i}' for i in left_sites+right_sites}
    )
    
    cross_term = (
        rho_lr.reindex(reindex_map)
        & rho_l
        & rho_r
    )
    cross_term = cross_term.contract()

    out = jnp.real(
        (purity_lr + purity_l*purity_r - 2*cross_term)/
        purity_lr
    )

    return out

## Extract cut rho and EDM

In [48]:
def generate_antilinear_edm_from_cut_state(cut_state, sites, num_defect_sites, t_inv=False):
    # Lots of duplicate code, probably a better way to do this.
    # Computes for U(t) or U(t^3)=U(t^-1), the two cases distinguished by t_inv.
    assert 2*num_defect_sites < len(sites)
    assert (num_defect_sites%2) == 0
    assert (len(sites)%2) == 0
    assert (sites[0]%2) == 0

    left_defect_sites = sites[:num_defect_sites]
    right_defect_sites = sites[-num_defect_sites:]
    internal_sites = sites[num_defect_sites:-num_defect_sites]

    if t_inv:
        unitary_part = qu_M.conj()
    else:
        unitary_part = qu_M

    # Being sloppy with the gate indices as the symmetries are invariant
    # under transpose and conjugation.
    left_sym_gates = [
        unitary_part.reindex({
            'ks': f'b{i}',
            'kf': f'b{i+1}',
            'bs': f'c{i}',
            'bf': f'c{i+1}'
        })
        for i in left_defect_sites[::2]
    ]

    inner_gates = [
        unitary_part.reindex({
            'ks': f'k{i}',
            'kf': f'k{i+1}',
            'bs': f'b{i}',
            'bf': f'b{i+1}'
        })
        for i in internal_sites[::2]
    ]

    right_sym_gates = [
        unitary_part.reindex({
            'ks': f'b{i}',
            'kf': f'b{i+1}',
            'bs': f'c{i}',
            'bf': f'c{i+1}'
        })
        for i in right_defect_sites[::2]
    ]

    reindex_map = (
        {
            f'k{i}': f'c{i}'
            for i in (left_defect_sites + right_defect_sites)
        }
        |
        {
            f'k{i}': f'b{i}'
            for i in internal_sites
        }
    )

    # The fact that we don't conjugate the reindexed cut_state means that
    # we are effectively implementing local complex conjugation.
    edm = (
        cut_state
        & cut_state.reindex(reindex_map)
        & left_sym_gates
        & inner_gates
        & right_sym_gates
    )

    edm = edm.contract()

    fuse_maps = [
        ('k_left', (f'k{i}' for i in left_defect_sites)),
        ('b_left', (f'b{i}' for i in left_defect_sites)),
        ('k_right', (f'k{i}' for i in right_defect_sites)),
        ('b_right', (f'b{i}' for i in right_defect_sites))
    ]

    edm.fuse(fuse_maps, inplace=True)

    return edm

In [49]:
def generate_fp_edm_from_cut_state(cut_state, sites, num_defect_sites):
    # Lots of duplicate code, probably a better way to do this.
    # Should really combine with generate_antilinear_edm_from_cut_state
    # Computes for U(t^2)=FP.
    assert 2*num_defect_sites < len(sites)
    assert (num_defect_sites%2) == 0
    assert (len(sites)%2) == 0
    assert (sites[0]%2) == 0

    left_defect_sites = sites[:num_defect_sites]
    right_defect_sites = sites[-num_defect_sites:]
    internal_sites = sites[num_defect_sites:-num_defect_sites]

    unitary_part = qu_spin_fermion_fp

    # Being sloppy with the gate indices as the symmetries are invariant
    # under transpose and conjugation.
    left_sym_gates = [
        unitary_part.reindex({
            'ks': f'b{i}',
            'kf': f'b{i+1}',
            'bs': f'c{i}',
            'bf': f'c{i+1}'
        })
        for i in left_defect_sites[::2]
    ]

    inner_gates = [
        unitary_part.reindex({
            'ks': f'k{i}',
            'kf': f'k{i+1}',
            'bs': f'b{i}',
            'bf': f'b{i+1}'
        })
        for i in internal_sites[::2]
    ]

    right_sym_gates = [
        unitary_part.reindex({
            'ks': f'b{i}',
            'kf': f'b{i+1}',
            'bs': f'c{i}',
            'bf': f'c{i+1}'
        })
        for i in right_defect_sites[::2]
    ]

    reindex_map = (
        {
            f'k{i}': f'c{i}'
            for i in (left_defect_sites + right_defect_sites)
        }
        |
        {
            f'k{i}': f'b{i}'
            for i in internal_sites
        }
    )

    edm = (
        cut_state
        & cut_state.conj().reindex(reindex_map)
        & left_sym_gates
        & inner_gates
        & right_sym_gates
    )

    edm = edm.contract()

    fuse_maps = [
        ('k_left', (f'k{i}' for i in left_defect_sites)),
        ('b_left', (f'b{i}' for i in left_defect_sites)),
        ('k_right', (f'k{i}' for i in right_defect_sites)),
        ('b_right', (f'b{i}' for i in right_defect_sites))
    ]

    edm.fuse(fuse_maps, inplace=True)

    return edm

In [50]:
def generate_all_edms_from_cut_state(cut_state, sites, num_defect_sites):
    args = (cut_state, sites, num_defect_sites)

    out = [
        generate_antilinear_edm_from_cut_state(*args, False),
        generate_fp_edm_from_cut_state(*args),
        generate_antilinear_edm_from_cut_state(*args, True)
    ]

    return out

## Defect operators

In [51]:
def random_uniform_complex(shape):
    return np.random.uniform(size=shape) + 1j*np.random.uniform(size=shape)

In [52]:
def solve_for_boundary_operators(edm, num_iters=100):
    # Careful, the indices are reversed here for ease.
    # i.e. the k, b indices have been swapped to make tensor contraction easier.
    scores = list()

    u_left = qtn.tensor_builder.rand_tensor(
        (edm.ind_size('b_left'), edm.ind_size('k_left')),
        inds=['k_left', 'b_left'],
        dtype='complex64'
    )

    u_right = qtn.tensor_builder.rand_tensor(
        (edm.ind_size('b_right'), edm.ind_size('k_right')),
        inds=['k_right', 'b_right'],
        dtype='complex64'
    )

    for _ in range(num_iters):
        right_edm = (edm & u_left).contract()
        data = right_edm.data
        U, S, VH = np.linalg.svd(data)
        scores.append(np.sum(S))
    
        sol = (U @ VH).conj().T
        u_right = qtn.Tensor(sol, inds = ['b_right', 'k_right'])

        left_edm = (edm & u_right).contract()
        data = left_edm.data
        U, S, VH = np.linalg.svd(data)
        scores.append(np.sum(S))
    
        sol = (U @ VH).conj().T
        u_left = qtn.Tensor(sol, inds = ['b_left', 'k_left'])

    return (u_left, u_right), scores

## Apply random unitary to groundstate

In [53]:
def generate_random_su2():
    # Randomly sample a unitary, and scale by the determinant.
    u = unitary_group.rvs(2)
    det_u = np.linalg.det(u)
    su = u*np.power(det_u, -0.5)

    return su

In [54]:
X =  generate_random_su2()

In [55]:
np.linalg.det(X)

np.complex128(1.0000000000000002-2.775557561562892e-16j)

In [56]:
np.round(X @ (X.conj().T), 3)

array([[ 1.+0.j, -0.+0.j],
       [-0.-0.j,  1.+0.j]])

In [57]:
def generate_random_symmetry_respecting_unitary_no_offset():
    # Generate a unitary which commutes with MK, where
    # M = X tensor (|0><0|) + Y tensor (|1><1|)

    phi = np.random.uniform(0, 2*np.pi)
    phi_phasor = np.exp(1j*phi)
    u0 = np.diag([phi_phasor, phi_phasor.conj()])

    if random() > 0.5:
        u0 = u0 @ np_X

    u1 = generate_random_su2()

    u = (
        tensor_product_operators(u0, np_00)
        + tensor_product_operators(u1, np_11)
    )

    qu_u = qtn.Tensor(
        u,
        inds=['ks', 'bs', 'kf', 'bf']
    )
    
    return qu_u

In [58]:
def generate_random_symmetry_respecting_unitary_offset():
    # Generate a unitary U such that IUI commutes with (MM)K, where
    # M = X tensor (|0><0|) + Y tensor (|1><1|), and concatenation denotes
    # tensor product.
    phi = np.random.uniform(0, 2*np.pi)
    phi_phasor = np.exp(1j*phi)
    u0 = np.diag([phi_phasor, phi_phasor.conj()])

    phi = np.random.uniform(0, 2*np.pi)
    phi_phasor = np.exp(1j*phi)
    u1 = np.diag([phi_phasor, phi_phasor.conj()])

    u = (
        tensor_product_operators(np_00, u0)
        + tensor_product_operators(np_11, u1)
    )

    qu_u = qtn.Tensor(
        u,
        inds=['kf', 'bf', 'ks', 'bs']
    )
    
    return qu_u

In [59]:
def generate_random_symmetry_respecting_unitary(offset):
    if offset:
        return generate_random_symmetry_respecting_unitary_offset()
    else:
        return generate_random_symmetry_respecting_unitary_no_offset()

In [60]:
# Warning, likely making assupmtions about shape of psi, number of sites being even here etc.
def apply_haar_random_fdlu_to_quimb_state(psi, domains_dict):
    num_sites = domains_dict['num_system_sites']

    depth = domains_dict['fdlu_depth']
    offset = domains_dict['fdlu_offset']
    all_circuit_lists = [
        list() for _ in range(depth)
    ]

    for layer, circuit_list in enumerate(all_circuit_lists):
        delta = layer
        is_offset = ((offset + delta)%2 == 1)

        for i in range(num_sites//2):
            site_1 = ((2*i)+delta+offset)%num_sites
            site_2 = ((2*i)+1+delta+offset)%num_sites

            u = generate_random_symmetry_respecting_unitary(is_offset)

            if is_offset:
                reindex_map = {
                    'kf': f'k_{layer+1}_{site_1}',
                    'ks': f'k_{layer+1}_{site_2}',
                    'bf': f'k_{layer}_{site_1}',
                    'bs': f'k_{layer}_{site_2}'
                }
            else:
                reindex_map = {
                    'ks': f'k_{layer+1}_{site_1}',
                    'kf': f'k_{layer+1}_{site_2}',
                    'bs': f'k_{layer}_{site_1}',
                    'bf': f'k_{layer}_{site_2}'
                }
            
            qu_u = u.reindex(reindex_map)

            circuit_list.append(qu_u)

    all_tensors = (
        [psi.reindex({f'k{i}': f'k_0_{i}' for i in range(num_sites)})]
        + sum(all_circuit_lists, start=[])
    )

    out = (
        qtn
        .TensorNetwork(all_tensors, virtual=False)
        .mangle_inner_()
        .reindex({f'k_{depth}_{i}': f'k{i}' for i in range(num_sites)}) 
    )

    return out

In [61]:
def extract_time_reversal_information_after_random_fdlu(psi, domains_dict,
    num_random_states=20):

    out = list()

    for _ in range(num_random_states):
        rand_psi = apply_haar_random_fdlu_to_quimb_state(psi, domains_dict)
        out.append(extract_time_reversal_information(rand_psi, domains_dict))

    return out

In [62]:
def extract_factorization_time_reversal_information_after_random_fdlu(psi,
    domains_dict, num_random_states=20):
    out = list()

    for _ in range(num_random_states):
        rand_psi = apply_haar_random_fdlu_to_quimb_state(psi, domains_dict)
        data = extract_factorization_time_reversal_information(
            rand_psi,
            domains_dict
        )
        out.append(data)

    return out

In [63]:
def get_quimb_psi_from_quspin_psi(quspin_psi):
    quimb_psi = qtn.Tensor(
        quspin_psi[::-1].reshape((2,)*num_sites),
        inds=[f'k{i}' for i in range(num_sites)]
    )

    return quimb_psi

## Sweep function

In [64]:
def extract_projector(psi, rho_sites, num_pad_sites, jw_even=False):
    proj_sites = rho_sites
    left_sites = list(range(
        min(rho_sites) - num_pad_sites,
        min(rho_sites)
    ))
    right_sites = list(range(
        max(rho_sites)+1,
        max(rho_sites)+num_pad_sites+1
    ))
    all_sites = left_sites + proj_sites + right_sites
    rho = (psi & psi.conj().reindex({f'k{i}': f'b{i}' for i in all_sites}))
    raw_proj_vec, proj_vec, losses = maximize_projector_states(
        rho,
        left_sites,
        proj_sites,
        right_sites
    )

    schmidt_vals = projector_state_check(
        proj_vec,
        rho,
        left_sites
    )

    return raw_proj_vec, proj_vec, losses, schmidt_vals

In [65]:
def find_invariants_via_projectors_from_random_state(psi, domains_dict, jw_even=False):
    rand_psi = apply_haar_random_fdlu_to_quimb_state(psi, domains_dict)
    
    raw_left_proj_vec, left_proj_vec, *left_proj_vec_results = extract_projector(
        rand_psi,
        domains_dict['left_projector_sites'],
        domains_dict['num_projector_pad_sites'],
        jw_even
    )
    
    raw_right_proj_vec, right_proj_vec, *right_proj_vec_results = extract_projector(
        rand_psi,
        domains_dict['right_projector_sites'],
        domains_dict['num_projector_pad_sites'],
        jw_even
    )
    
    cut_sites = list(range(
        min(domains_dict['left_projector_sites']),
        max(domains_dict['right_projector_sites'])+1
    ))
    
    cut_rho_unprojected = (
        rand_psi
        & rand_psi.conj().reindex({f'k{i}': f'b{i}' for i in cut_sites})
    )
    
    left_proj_sites = domains_dict['left_projector_sites']
    right_proj_sites = domains_dict['right_projector_sites']
    
    cut_rho = (
        cut_rho_unprojected.reindex({
            **{f'k{i}': f'l{i}' for i in left_proj_sites},
            **{f'k{i}': f'l{i}' for i in right_proj_sites},
            **{f'b{i}': f'c{i}' for i in left_proj_sites},
            **{f'b{i}': f'c{i}' for i in right_proj_sites}
        })
        & left_proj_vec.conj().reindex({f'k{i}': f'l{i}' for i in left_proj_sites})
        & left_proj_vec
        & left_proj_vec.reindex({f'k{i}': f'c{i}' for i in left_proj_sites})
        & left_proj_vec.conj().reindex({f'k{i}': f'b{i}' for i in left_proj_sites})
        & right_proj_vec.conj().reindex({f'k{i}': f'l{i}' for i in right_proj_sites})
        & right_proj_vec
        & right_proj_vec.reindex({f'k{i}': f'c{i}' for i in right_proj_sites})
        & right_proj_vec.conj().reindex({f'k{i}': f'b{i}' for i in right_proj_sites})
    )
    
    cut_rho_trace = (
        cut_rho
        .reindex({f'b{i}': f'k{i}' for i in cut_sites})
        .contract()
    )
    
    cut_rho = cut_rho/cut_rho_trace
    
    tranpose_map = (
        {f'k{i}': f'b{i}' for i in cut_sites}
        | {f'b{i}': f'k{i}' for i in cut_sites}
    )
    
    cut_rho_purity = (
        (cut_rho & cut_rho.reindex(tranpose_map))
        .contract()
    )
    
    sub_cut_sites = list(range(
        max(domains_dict['left_projector_sites'])+1,
        min(domains_dict['right_projector_sites'])
    ))
    
    sub_cut_rho = (
        cut_rho
        .reindex({f'k{i}': f'b{i}' for i in left_proj_sites + right_proj_sites})
        #.contract()
    )
    
    sub_cut_rho_trace = (
        sub_cut_rho
        .reindex({f'b{i}': f'k{i}' for i in sub_cut_sites})
        .contract()
    )
    
    tranpose_map = (
        {f'k{i}': f'b{i}' for i in sub_cut_sites}
        | {f'b{i}': f'k{i}' for i in sub_cut_sites}
    )
    
    sub_cut_rho_purity = (
        (sub_cut_rho & sub_cut_rho.reindex(tranpose_map))
        .contract()
    )
    
    cut_score, cut_state, cut_overlap = get_dominant_eigenvector(sub_cut_rho, False)

    cut_state_fermion_parity = compute_fermion_parity(cut_state, sub_cut_sites)

    edms = generate_all_edms_from_cut_state(
        cut_state,
        sub_cut_sites,
        domains_dict['num_defect_sites']
    )

    defect_ops_results = [
        solve_for_boundary_operators(
            edm,
            num_iters=20
        )
        for edm in edms
    ]

    left_defect_sites = list(range(
        max(domains_dict['left_projector_sites']) + 1,
        max(domains_dict['left_projector_sites']) + 1 + domains_dict['num_defect_sites']
    ))
    right_defect_sites = list(range(
        min(domains_dict['right_projector_sites']) - domains_dict['num_defect_sites'],
        min(domains_dict['right_projector_sites'])
    ))
    
    left_rdm = (
        cut_rho
        .reindex({
            f'b{i}': f'k{i}'
            for i in cut_sites if i not in left_defect_sites
        })
    )
    left_rdm = left_rdm.contract()
    left_fuse_map = [
        ('k_left', [f'k{i}' for i in left_defect_sites]),
        ('b_left', [f'b{i}' for i in left_defect_sites])
    ]
    left_rdm.fuse(left_fuse_map, inplace=True)
    
    right_rdm = (
        cut_rho
        .reindex({
            f'b{i}': f'k{i}'
            for i in cut_sites if i not in right_defect_sites
        })
    )
    right_rdm = right_rdm.contract()
    right_fuse_map = [
        ('k_right', [f'k{i}' for i in right_defect_sites]),
        ('b_right', [f'b{i}' for i in right_defect_sites])
    ]
    right_rdm.fuse(right_fuse_map, inplace=True)

    defect_ops_pairs = [l[0] for l in defect_ops_results]
    left_defect_ops, right_defect_ops = list(zip(*defect_ops_pairs))

    np_left_rdm = (
        left_rdm
        .transpose('k_left', 'b_left')
        .data
    )

    np_left_defect_ops = [
        left_defect_op.transpose('k_left', 'b_left').data.T
        for left_defect_op in left_defect_ops
    ]

    np_right_rdm = (
        right_rdm
        .transpose('k_right', 'b_right')
        .data
    )
    
    np_right_defect_ops = [
        right_defect_op.transpose('k_right', 'b_right').data.T
        for right_defect_op in right_defect_ops
    ]

    np_left_defect_ops = [
        np.identity(np_left_defect_ops[0].shape[0]),
        *np_left_defect_ops
    ]
    np_right_defect_ops = [
        np.identity(np_right_defect_ops[0].shape[0]),
        *np_right_defect_ops
    ]

    """
    #Ignore for now
    if domains_dict['normalize_defect_op']:
        np_left_defect_op = normalize_defect_op(
            np_left_defect_op,
            np_left_rdm
        )
        right_defect_op = normalize_defect_op(
            np_right_defect_op,
            np_right_rdm
        )
    """

    fp_list = [np_I, np_Z]
    np_fp = multikron([
        fp_list[i%2]
        for i in range(domains_dict['num_defect_sites'])
    ])

    left_defect_op_invariant = np.trace(
        np_fp
        @ np_left_defect_ops[1].conj().T
        @ np_fp
        @ np_left_defect_ops[1]
        @ np_left_rdm
    )

    right_defect_op_invariant = np.trace(
        np_fp
        @ np_right_defect_ops[1].conj().T
        @ np_fp
        @ np_right_defect_ops[1]
        @ np_right_rdm
    )

    np_sym_defect_m = get_multisite_op(
        np_M_reindexed,
        domains_dict['num_defect_sites']
    )

    np_defect_I = multikron([
        np_I
        for i in range(domains_dict['num_defect_sites'])
    ])

    unitary_sym_op_list = [
        np_defect_I,
        np_sym_defect_m,
        np_fp,
        np_sym_defect_m.conj()
    ]

    left_cocyles = get_cocycles_from_rho(
        np_left_rdm,
        np_left_defect_ops,
        unitary_sym_op_list
    )
    left_cocyles = gauge_fix_cocycles(left_cocyles)

    right_cocyles = get_cocycles_from_rho(
        np_right_rdm,
        np_right_defect_ops,
        unitary_sym_op_list
    )
    right_cocyles = gauge_fix_cocycles(right_cocyles)

    all_cocyles = np.array([left_cocyles, right_cocyles])

    cocycle_equation_output = compute_cocyle_equation_from_cocycles(all_cocyles)
    bosonic_cocycle_equation_output = (
        compute_bosonic_cocyle_equation_from_cocycles(all_cocyles)
    )

    cocycle_invariants = (all_cocyles[:,0,0]**2)*(all_cocyles[:,1,1])

    fermion_decoration_from_cocycle = all_cocyles[:, 0, 1]/all_cocyles[:, 1, 0]
    """
    left_bosonic_cocycle = get_bosonic_cocycle(
        np_left_rdm,
        np_left_defect_ops[1],
        unitary_sym_op_list
    )
    right_bosonic_cocycle = get_bosonic_cocycle(
        np_right_rdm,
        np_right_defect_ops[1],
        unitary_sym_op_list
    )

    left_bosonic_cocyle_eq_from_boosnic_cocycle = (
        compute_bosonic_cocyle_equation_from_bosonic_cocycle(
            left_bosonic_cocycle
        )
    )
    right_bosonic_cocyle_eq_from_boosnic_cocycle = (
        compute_bosonic_cocyle_equation_from_bosonic_cocycle(
            right_bosonic_cocycle
        )
    )
    """

    out = {
        'left_proj_vec': left_proj_vec,
        'raw_left_proj_vec': raw_left_proj_vec,
        'left_proj_vec_results': left_proj_vec_results,
        'right_proj_vec': right_proj_vec,
        'raw_right_proj_vec': raw_right_proj_vec,
        'right_proj_vec_results': right_proj_vec_results,
        'cut_rho_trace': cut_rho_trace,
        'cut_rho_purity': cut_rho_purity,
        'sub_cut_rho_trace': sub_cut_rho_trace,
        'sub_cut_rho_purity': sub_cut_rho_purity,
        'cut_score': cut_score,
        'cut_state': cut_state,
        'cut_state_fermion_parity': cut_state_fermion_parity,
        'cut_overlap': cut_overlap,
        'defect_ops_scores': [l[1] for l in defect_ops_results],
        'left_defect_ops': left_defect_ops,
        'right_defect_ops': right_defect_ops,
        'left_defect_op_invariant': left_defect_op_invariant,
        'right_defect_op_invariant': right_defect_op_invariant,
        'cocycles': all_cocyles,
        'cocycle_equation_output': cocycle_equation_output,
        'bosonic_cocycle_equation_output': bosonic_cocycle_equation_output,
        'cocycle_invariants': cocycle_invariants,
        'fermion_decoration_from_cocycle': fermion_decoration_from_cocycle
    }

    return out

## Find dominant eigenvector

In [66]:
def random_uniform_complex(shape):
    return np.random.uniform(size=shape) + 1j*np.random.uniform(size=shape)

In [67]:
def lanczos_iteration(rho, v, sites):
    w = (
        rho & v.reindex({f'k{i}': f'b{i}' for i in sites})
    ).contract()

    w_norm = np.sqrt((w & w.conj()).contract())

    out = w/w_norm

    return out

In [68]:
def multiply_state_by_jw_string(state, sites):
    gates = [
        qu_Z.reindex({'k': f'k{i}', 'b': f'b{i}'})
        for i in sites if (i%2 == 1)
    ]

    out = qtn.TensorNetwork(
        [
            *gates,
            state.reindex({f'k{i}': f'b{i}' for i in sites if (i%2 == 1)})
        ]
    )

    return out.contract()

In [69]:
def lanczos_iteration_jw_even(rho, v, sites):
    w = multiply_state_by_jw_string(v, sites)
    w = (v+w)/2

    w = (
        rho & w.reindex({f'k{i}': f'b{i}' for i in sites})
    ).contract()

    w = (multiply_state_by_jw_string(w, sites) + w)/2

    w_norm = np.sqrt((w & w.conj()).contract())

    out = w/w_norm

    return out

In [70]:
def lanczos_algorithm(rho, v, sites, num_iters=20, jw_even=False):
    update_func = lanczos_iteration_jw_even if jw_even else lanczos_iteration
    for _ in range(num_iters):
        v = update_func(rho, v, sites)

    return v

In [71]:
def get_dominant_eigenvector(rho, jw_even=False):
    k_inds = [i for i in rho.outer_inds() if i.startswith('k')]
    #b_inds = [i for i in rho.inds if i.startswith('b')]

    sites = [int(s[1:]) for s in k_inds]

    v = qtn.Tensor(
        data=random_uniform_complex((2,)*len(sites)),
        inds=[f'k{i}' for i in sites]
    )

    v = lanczos_algorithm(
        rho,
        v,
        sites,
        num_iters=20,
        jw_even=jw_even
    )

    update_func = lanczos_iteration_jw_even if jw_even else lanczos_iteration
    new_v = update_func(rho, v, sites)

    overlap = np.abs((v & new_v.conj()).contract())

    score = (
        v.reindex({f'k{i}': f'b{i}' for i in sites})
        & rho
        & v.conj()
    )
    score = score.contract()


    return score, v, overlap

In [72]:
def compute_fermion_parity(state, sites):
    odd_sites = [i for i in sites if (i%2 == 1)]

    gates = [
        qu_Z.reindex({'k': f'k{i}', 'b': f'b{i}'})
        for i in odd_sites
    ]

    gate_exp = (
        state.reindex({f'k{i}': f'b{i}' for i in odd_sites})
        & gates
        & state.conj()
    ).contract()

    return gate_exp

## Extract cocyles

In [73]:
def get_multisite_op(op, num_defect_sites):
    assert (num_defect_sites%2 == 0)

    np_op = multikron([op,]*(num_defect_sites//2))

    out = qtn.Tensor(
        np_op,
        inds=['k', 'b']
    )

    out = out.transpose('k', 'b').data

    return out

In [74]:
def normalize_defect_op(defect_op, defect_rho):
    exp = np.trace(
        defect_op
        @ defect_op
        @ defect_rho
    )

    phase = np.power(exp/np.abs(exp), -0.5)

    out = defect_op*phase

    return out

In [75]:
def get_t_squared_defect_op_from_t_defect_op(defect_op, unitary_sym_op):
    out = (
        unitary_sym_op
        @ (defect_op.conj())
        @ (unitary_sym_op.conj().T)
        @ defect_op
    )

    return out

In [76]:
def get_t_inverse_defect_op_from_t_defect_op(defect_op, unitary_sym_op):
    out = (
        (unitary_sym_op.T)
        @ (defect_op.T)
        @ (unitary_sym_op.conj())
    )

    return out

In [77]:
def get_all_defect_ops_from_t_defect_op(defect_op, unitary_sym_op):
    out = [
        np.identity(defect_op.shape[0]),
        defect_op,
        get_t_squared_defect_op_from_t_defect_op(defect_op, unitary_sym_op),
        get_t_inverse_defect_op_from_t_defect_op(defect_op, unitary_sym_op)
    ]

    return out

In [78]:
def get_cocycle(defect_rho, defect_op_list, unitary_sym_op_list, g_index,
                  h_index):
    g_op = defect_op_list[g_index]

    h_op = defect_op_list[h_index]
    h_op = h_op.conj() if (g_index%2 == 1) else h_op

    gh_index = (g_index+h_index)%4
    gh_op = defect_op_list[gh_index]

    unitary_sym_op = unitary_sym_op_list[g_index]

    out = np.trace(
        (gh_op.conj().T)
        @ unitary_sym_op
        @ h_op
        @ (unitary_sym_op.conj().T)
        @ g_op
        @ defect_rho
    )

    return out

In [79]:
def get_cocycles_from_rho(defect_rho, defect_op_list, unitary_sym_op_list):
    out = np.array([
        [
            get_cocycle(defect_rho, defect_op_list, unitary_sym_op_list, i, j)
            for j in range(1,4)
        ]
        for i in range(1,4)
    ])

    return out

In [80]:
def gauge_fix_cocycles(cocycles):
    # Gauge fix cocycles to match appendix A.2 of Turzillo & You 2019
    L = np.array([
        1,
        1,
        np.power(cocycles[1,1], 0.5),
        np.power(cocycles[1,1], 0.5)/cocycles[1,0]
    ])

    gauge_transformation = np.ones((3,3), dtype=complex)

    for g, h in product(range(1,4), repeat=2):
        num = L[(g+h)%4]
        if (g%2 == 1):
            denom = (L[h].conj())*L[g]
        else:
            denom = L[h]*L[g]
        gauge_transformation[(g-1), (h-1)] = num/denom

    out = cocycles*gauge_transformation

    return out

In [81]:
def compute_cocyle_equation_from_cocycles(cocyles):
    shape = cocyles
    # A bit janky, oh well
    X = np.ones((2, 4, 4), dtype=complex)

    X[:, 1:, 1:] = cocyles

    out = np.zeros((2, 4, 4, 4), dtype=complex)

    for i,j,k in product(range(4), repeat=3):
        num = X[:,i,j]*X[:,(i+j)%4, k]
        if (i%2 == 0):
            denom = X[:,j,k]*X[:, i, (j+k)%4]
        if (i%2 == 1):
            denom = (X[:,j,k].conj())*X[:,i,(j+k)%4]

        #print(num)
        #print(denom)

        out[:,i,j,k] = num/denom

    return out

In [82]:
def compute_bosonic_cocyle_equation_from_cocycles(cocyles):
    bosonic_cocycles = cocyles[..., :1, :1]
    shape = bosonic_cocycles
    # A bit janky, oh well
    X = np.ones((2, 2, 2), dtype=complex)

    X[:, 1:, 1:] = bosonic_cocycles

    out = np.zeros((2, 2, 2, 2), dtype=complex)

    for i,j,k in product(range(2), repeat=3):
        num = X[:,i,j]*X[:,(i+j)%2, k]
        if (i == 0):
            denom = X[:,j,k]*X[:, i, (j+k)%2]
        if (i == 1):
            denom = (X[:,j,k].conj())*X[:,i,(j+k)%2]

        out[:,i,j,k] = num/denom

    return out

In [83]:
def get_bosonic_cocycle(defect_rho, defect_op, unitary_sym_op_list):
    defect_op_list = get_all_defect_ops_from_t_defect_op(
        defect_op,
        unitary_sym_op_list[1]
    )

    defect_op = defect_op_list[1]
    fp_defect_op = defect_op_list[2]

    unitary_sym_op = unitary_sym_op_list[1]
    fp_sym_op = unitary_sym_op_list[2]

    out = np.trace(
        (fp_defect_op.conj().T)
        @ unitary_sym_op
        @ defect_op.conj()
        @ (unitary_sym_op.conj().T)
        @ defect_op
        @ defect_rho
    )

    return out

In [84]:
def compute_bosonic_cocyle_equation_from_bosonic_cocycle(bosonic_cocycle):
    X = np.ones((2, 2), dtype=complex)

    X[1, 1] = bosonic_cocycle

    out = np.zeros((2, 2, 2), dtype=complex)

    for i,j,k in product(range(2), repeat=3):
        num = X[i,j]*X[(i+j)%2, k]
        if (i == 0):
            denom = X[j,k]*X[i, (j+k)%2]
        if (i == 1):
            denom = (X[j,k].conj())*X[i,(j+k)%2]

        out[i,j,k] = num/denom

    return out

# Check random unitaries and symmetry

In [85]:
domains_dict = {
    'num_system_sites': 24,
    'left_projector_sites': list(range(4, 8)),
    'right_projector_sites': list(range(16, 20)),
    'num_projector_pad_sites': 2,
    'num_defect_sites': 2,
    'fdlu_depth': 2,
    'fdlu_offset': 1
}

In [86]:
cluster_psi = get_cluster_state_qu_tensor_network(domains_dict['num_system_sites'])

In [87]:
rand_psi = apply_haar_random_fdlu_to_quimb_state(cluster_psi, domains_dict)

In [88]:
(rand_psi & rand_psi.conj()).contract()

np.complex128(0.9999999999999974+1.3877787807814457e-17j)

In [89]:
(rand_psi & cluster_psi.conj()).contract()

np.complex128(-4.42451097315421e-05-1.1519648082658485e-19j)

In [90]:
symmetry_gates = [
    qu_M.reindex({
        'ks': f'k{i}', 'bs':f'b{i}',
        'kf': f'k{i+1}', 'bf':f'b{i+1}',
    })
    for i in range(0, domains_dict['num_system_sites'], 2)
]

In [91]:
sym_cluster_psi = qtn.TensorNetwork(
    [
        rand_psi.reindex({f'k{i}': f'b{i}' for i in range(domains_dict['num_system_sites'])}),
        *symmetry_gates
    ]
)

In [92]:
(
    sym_cluster_psi & sym_cluster_psi.conj()
).contract()

np.complex128(0.9999999999999973-2.7755575615628914e-17j)

In [93]:
(
    sym_cluster_psi & rand_psi.conj()
).contract()

np.complex128(1.7532932320450125e-05+2.303929616531697e-19j)

In [94]:
(
    sym_cluster_psi & rand_psi
).contract()

np.complex128(0.999999999999997-6.938893903907228e-17j)

In [95]:
product_psi = get_product_qu_tensor_network(domains_dict['num_system_sites'])

In [96]:
rand_psi = apply_haar_random_fdlu_to_quimb_state(product_psi, domains_dict)

In [97]:
(rand_psi & rand_psi.conj()).contract()

np.complex128(0.9999999999999977+1.8041124150158794e-16j)

In [98]:
(rand_psi & product_psi.conj()).contract()

np.complex128(-0.008886181063620964-3.004793622026769e-18j)

In [99]:
symmetry_gates = [
    qu_M.reindex({
        'ks': f'k{i}', 'bs':f'b{i}',
        'kf': f'k{i+1}', 'bf':f'b{i+1}',
    })
    for i in range(0, domains_dict['num_system_sites'], 2)
]

In [100]:
sym_product_psi = qtn.TensorNetwork(
    [
        rand_psi.reindex({f'k{i}': f'b{i}' for i in range(domains_dict['num_system_sites'])}),
        *symmetry_gates
    ]
)

In [101]:
(
    sym_product_psi & sym_product_psi.conj()
).contract()

np.complex128(0.9999999999999977+1.8041124150158794e-16j)

In [102]:
(
    sym_product_psi & rand_psi.conj()
).contract()

np.complex128(-0.0001945115868219867+1.212951180468158e-18j)

In [103]:
(
    sym_product_psi & rand_psi
).contract()

np.complex128(0.9999999999999977+1.8041124150158794e-16j)

So random unitaries are symmetric, nice.

In [104]:
import cProfile
import pstats
from time import perf_counter

prof = cProfile.Profile()
start = perf_counter()

prof.enable()
try:
    current = find_invariants_via_projectors_from_random_state(
        cluster_psi, domains_dict, jw_even=True
    )
finally:
    prof.disable()
    print(f"Elapsed: {perf_counter() - start:.2f} s")
    pstats.Stats(prof).strip_dirs().sort_stats("cumulative").print_stats(30)

print("Projector loss evaluations:",
      len(current["left_proj_vec_results"][0]),
      len(current["right_proj_vec_results"][0]))

Elapsed: 2.36 s
         2406547 function calls (2346788 primitive calls) in 2.367 seconds

   Ordered by: cumulative time
   List reduced from 3130 to 30 due to restriction <30>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
    85/22    0.000    0.000    1.506    0.068 traceback_util.py:191(reraise_with_filtered_traceback)
    61/12    0.000    0.000    1.504    0.125 pjit.py:243(cache_miss)
    61/58    0.001    0.000    1.198    0.021 pjit.py:128(_run_python_pjit)
       12    0.000    0.000    1.135    0.095 pjit.py:1124(_pjit_call_impl_python)
   396/48    0.001    0.000    1.120    0.023 profiler.py:424(wrapper)
        2    0.000    0.000    1.083    0.542 optimize.py:1488(optimize)
        2    0.000    0.000    1.083    0.542 optimize.py:1526(optimize_scipy)
        2    0.000    0.000    0.990    0.495 _minimize.py:54(minimize)
        2    0.000    0.000    0.989    0.495 _lbfgsb_py.py:294(_minimize_lbfgsb)
       68    0.000    0.000    0.986    0

# Cluster state - depth 2

In [105]:
domains_dict = {
    'num_system_sites': 32,
    'left_projector_sites': list(range(4, 8)),
    'right_projector_sites': list(range(22, 28)),
    'num_projector_pad_sites': 2,
    'num_defect_sites': 4,
    'fdlu_depth': 2,
    'fdlu_offset': 0
}

In [106]:
cluster_psi = get_cluster_state_qu_tensor_network(domains_dict['num_system_sites'])

In [107]:
results = list()

for _ in tqdm(range(1000)):
    current = find_invariants_via_projectors_from_random_state(
        cluster_psi,
        domains_dict,
        jw_even=True
    )
    results.append(current)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [22:53<00:00,  1.37s/it]


In [109]:
with open(r"results/z_4_tf_cluster_results.pkl", "wb") as file:
    pickle.dump(results, file)

### Analyze results

#### Projector scores

In [110]:
np.round(np.array([
    d['left_proj_vec_results'][0][-1]
    for d in results
]), 3)

array([-0.,  0., -0., -0.,  0., -0., -0., -0.,  0., -0., -0., -0., -0.,
       -0.,  0., -0.,  0., -0., -0.,  0., -0.,  0.,  0.,  0., -0.,  0.,
       -0., -0.,  0., -0., -0.,  0., -0.,  0., -0.,  0.,  0., -0., -0.,
        0., -0.,  0.,  0.,  0.,  0., -0.,  0.,  0., -0.,  0.,  0., -0.,
       -0.,  0.,  0.,  0., -0.,  0., -0., -0., -0., -0., -0., -0.,  0.,
        0., -0.,  0.,  0.,  0., -0.,  0.,  0., -0., -0.,  0.,  0.,  0.,
       -0.,  0., -0.,  0.,  0.,  0.,  0., -0., -0., -0.,  0.,  0.,  0.,
        0.,  0., -0.,  0., -0.,  0., -0.,  0.,  0., -0.,  0., -0.,  0.,
        0.,  0.,  0., -0.,  0., -0., -0.,  0.,  0.,  0., -0., -0., -0.,
        0., -0., -0.,  0., -0.,  0.,  0., -0.,  0.,  0., -0.,  0.,  0.,
        0.,  0.,  0., -0., -0., -0., -0., -0.,  0.,  0., -0.,  0., -0.,
        0.,  0.,  0., -0.,  0., -0.,  0., -0., -0.,  0.,  0.,  0.,  0.,
        0.,  0.,  0.,  0.,  0.,  0.,  0., -0., -0.,  0.,  0., -0., -0.,
        0., -0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0

In [111]:
np.round(np.array([
    d['right_proj_vec_results'][0][-1]
    for d in results
]), 3)

array([-0., -0.,  0., -0.,  0., -0.,  0.,  0.,  0.,  0.,  0.,  0., -0.,
        0.,  0., -0.,  0.,  0., -0., -0., -0., -0.,  0., -0.,  0.,  0.,
       -0.,  0.,  0., -0.,  0.,  0.,  0.,  0.,  0., -0.,  0.,  0.,  0.,
        0., -0., -0., -0., -0.,  0., -0.,  0., -0., -0.,  0.,  0.,  0.,
        0.,  0.,  0.,  0.,  0.,  0.,  0., -0.,  0.,  0.,  0.,  0.,  0.,
        0.,  0., -0.,  0.,  0.,  0.,  0.,  0.,  0., -0.,  0., -0., -0.,
       -0.,  0.,  0.,  0.,  0.,  0., -0., -0., -0.,  0., -0.,  0.,  0.,
        0., -0.,  0.,  0.,  0., -0., -0., -0.,  0.,  0.,  0., -0.,  0.,
       -0.,  0., -0., -0.,  0.,  0.,  0., -0., -0.,  0.,  0.,  0.,  0.,
       -0.,  0.,  0.,  0., -0.,  0.,  0.,  0.,  0.,  0.,  0.,  0., -0.,
        0.,  0.,  0.,  0., -0., -0.,  0.,  0.,  0.,  0., -0.,  0., -0.,
        0.,  0.,  0., -0., -0.,  0.,  0.,  0.,  0., -0.,  0.,  0.,  0.,
        0., -0., -0., -0.,  0.,  0., -0., -0.,  0.,  0., -0.,  0.,  0.,
        0.,  0.,  0.,  0., -0.,  0.,  0., -0., -0.,  0.,  0.,  0

In [112]:
def get_schmidt_vals_ratio(schmidt_vals):
    if len(schmidt_vals.data) > 1:
        return schmidt_vals.data[1]/schmidt_vals.data[0]
    else:
        return 0

In [114]:
np.round(np.array([
    get_schmidt_vals_ratio(d['left_proj_vec_results'][1])
    for d in results
]), 3)

array([0.285, 0.103, 0.062, 0.059, 0.057, 0.603, 0.638, 0.003, 0.065,
       0.059, 0.264, 0.003, 0.106, 0.432, 0.314, 0.684, 0.625, 0.247,
       0.061, 0.304, 0.019, 0.333, 0.15 , 0.268, 0.203, 0.263, 0.11 ,
       0.396, 0.009, 0.249, 0.519, 0.098, 0.038, 0.49 , 0.093, 0.125,
       0.039, 0.054, 0.033, 0.533, 0.731, 0.207, 0.289, 0.127, 0.062,
       0.611, 0.547, 0.068, 0.603, 0.06 , 0.488, 0.   , 0.17 , 0.086,
       0.262, 0.042, 0.232, 0.113, 0.752, 0.114, 0.046, 0.016, 0.476,
       0.495, 0.068, 0.572, 0.685, 0.114, 0.095, 0.034, 0.039, 0.688,
       0.099, 0.008, 0.246, 0.504, 0.125, 0.048, 0.175, 0.623, 0.286,
       0.459, 0.315, 0.021, 0.064, 0.001, 0.261, 0.557, 0.02 , 0.068,
       0.217, 0.183, 0.082, 0.6  , 0.871, 0.146, 0.404, 0.245, 0.069,
       0.933, 0.274, 0.12 , 0.005, 0.265, 0.033, 0.047, 0.092, 0.008,
       0.557, 0.604, 0.188, 0.066, 0.135, 0.133, 0.072, 0.156, 0.541,
       0.17 , 0.071, 0.304, 0.294, 0.387, 0.128, 0.293, 0.117, 0.374,
       0.467, 0.184,

In [115]:
np.round(np.array([
    get_schmidt_vals_ratio(d['right_proj_vec_results'][1])
    for d in results
]), 3)

array([0.116, 0.534, 0.146, 0.442, 0.274, 0.276, 0.375, 0.342, 0.282,
       0.252, 0.501, 0.592, 0.376, 0.164, 0.699, 0.228, 0.226, 0.271,
       0.372, 0.791, 0.21 , 0.324, 0.948, 0.457, 0.036, 0.358, 0.373,
       0.661, 0.119, 0.367, 0.161, 0.068, 0.065, 0.839, 0.225, 0.19 ,
       0.32 , 0.429, 0.414, 0.395, 0.15 , 0.346, 0.163, 0.437, 0.492,
       0.23 , 0.27 , 0.613, 0.569, 0.182, 0.44 , 0.28 , 0.696, 0.242,
       0.532, 0.104, 0.274, 0.354, 0.247, 0.528, 0.118, 0.124, 0.203,
       0.725, 0.919, 0.204, 0.522, 0.492, 0.347, 0.399, 0.149, 0.486,
       0.45 , 0.121, 0.085, 0.396, 0.076, 0.22 , 0.37 , 0.213, 0.108,
       0.191, 0.262, 0.196, 0.199, 0.119, 0.072, 0.1  , 0.233, 0.513,
       0.338, 0.436, 0.604, 0.329, 0.092, 0.533, 0.357, 0.609, 0.585,
       0.07 , 0.383, 0.359, 0.101, 0.547, 0.579, 0.167, 0.076, 0.46 ,
       0.064, 0.099, 0.012, 0.396, 0.378, 0.763, 0.115, 0.07 , 0.257,
       0.305, 0.587, 0.232, 0.154, 0.485, 0.276, 0.355, 0.15 , 0.086,
       0.484, 0.203,

#### Purities

In [116]:
np.round(np.array(
    [d['cut_rho_purity'] for d in results]
), 3)

array([1.   +0.j, 1.   -0.j, 1.   -0.j, 1.   -0.j, 0.999-0.j, 0.999+0.j,
       1.   +0.j, 1.   +0.j, 1.   +0.j, 1.   +0.j, 0.999+0.j, 1.   -0.j,
       1.   +0.j, 1.   -0.j, 1.   +0.j, 1.   +0.j, 0.999-0.j, 1.   +0.j,
       1.   -0.j, 1.   -0.j, 1.   +0.j, 0.999+0.j, 1.   +0.j, 1.   -0.j,
       1.   +0.j, 1.   +0.j, 1.   -0.j, 1.   -0.j, 1.   -0.j, 1.   -0.j,
       1.   +0.j, 1.   -0.j, 1.   +0.j, 1.   -0.j, 1.   -0.j, 0.999-0.j,
       1.   +0.j, 1.   -0.j, 1.   -0.j, 1.   -0.j, 1.   -0.j, 1.   +0.j,
       1.   -0.j, 1.   +0.j, 1.   +0.j, 1.   +0.j, 1.   +0.j, 1.   -0.j,
       1.   -0.j, 0.997-0.j, 1.   -0.j, 1.   +0.j, 1.   +0.j, 1.   +0.j,
       1.   -0.j, 1.   -0.j, 1.   -0.j, 0.998+0.j, 1.   -0.j, 1.   +0.j,
       0.999+0.j, 1.   +0.j, 1.   -0.j, 1.   +0.j, 1.   +0.j, 1.   +0.j,
       1.   -0.j, 1.   +0.j, 1.   -0.j, 1.   +0.j, 1.   -0.j, 1.   +0.j,
       1.   +0.j, 1.   -0.j, 1.   +0.j, 1.   +0.j, 1.   +0.j, 1.   -0.j,
       1.   +0.j, 0.999-0.j, 1.   -0.j, 1.   +0.j, 

In [117]:
np.round(np.array(
    [d['sub_cut_rho_trace'] for d in results]
), 3)

array([1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j,
       1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j,
       1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j,
       1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.-0.j,
       1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j,
       1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j,
       1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j,
       1.-0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j,
       1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j,
       1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j,
       1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j,
       1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j,
       1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j,
       1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j,
      

In [118]:
np.round(np.array(
    [d['sub_cut_rho_purity'] for d in results]
), 3)

array([1.   +0.j, 1.   -0.j, 1.   -0.j, 1.   -0.j, 0.999-0.j, 0.999+0.j,
       1.   +0.j, 1.   +0.j, 1.   +0.j, 1.   +0.j, 0.999+0.j, 1.   -0.j,
       1.   +0.j, 1.   -0.j, 1.   +0.j, 1.   +0.j, 0.999-0.j, 1.   +0.j,
       1.   -0.j, 1.   -0.j, 1.   +0.j, 0.999+0.j, 1.   +0.j, 1.   -0.j,
       1.   +0.j, 1.   +0.j, 1.   -0.j, 1.   -0.j, 1.   -0.j, 1.   -0.j,
       1.   +0.j, 1.   -0.j, 1.   +0.j, 1.   -0.j, 1.   -0.j, 0.999-0.j,
       1.   +0.j, 1.   -0.j, 1.   -0.j, 1.   -0.j, 1.   -0.j, 1.   +0.j,
       1.   -0.j, 1.   +0.j, 1.   +0.j, 1.   +0.j, 1.   +0.j, 1.   -0.j,
       1.   -0.j, 0.997-0.j, 1.   -0.j, 1.   +0.j, 1.   +0.j, 1.   +0.j,
       1.   -0.j, 1.   -0.j, 1.   -0.j, 0.998+0.j, 1.   -0.j, 1.   +0.j,
       0.999+0.j, 1.   +0.j, 1.   -0.j, 1.   +0.j, 1.   +0.j, 1.   +0.j,
       1.   -0.j, 1.   +0.j, 1.   -0.j, 1.   +0.j, 1.   -0.j, 1.   +0.j,
       1.   +0.j, 1.   -0.j, 1.   +0.j, 1.   +0.j, 1.   +0.j, 1.   -0.j,
       1.   +0.j, 0.999-0.j, 1.   -0.j, 1.   +0.j, 

Purities are the same, which one could likely prove.

In [119]:
np.round(np.array(
    [d['sub_cut_rho_purity'] for d in results]
), 3)

array([1.   +0.j, 1.   -0.j, 1.   -0.j, 1.   -0.j, 0.999-0.j, 0.999+0.j,
       1.   +0.j, 1.   +0.j, 1.   +0.j, 1.   +0.j, 0.999+0.j, 1.   -0.j,
       1.   +0.j, 1.   -0.j, 1.   +0.j, 1.   +0.j, 0.999-0.j, 1.   +0.j,
       1.   -0.j, 1.   -0.j, 1.   +0.j, 0.999+0.j, 1.   +0.j, 1.   -0.j,
       1.   +0.j, 1.   +0.j, 1.   -0.j, 1.   -0.j, 1.   -0.j, 1.   -0.j,
       1.   +0.j, 1.   -0.j, 1.   +0.j, 1.   -0.j, 1.   -0.j, 0.999-0.j,
       1.   +0.j, 1.   -0.j, 1.   -0.j, 1.   -0.j, 1.   -0.j, 1.   +0.j,
       1.   -0.j, 1.   +0.j, 1.   +0.j, 1.   +0.j, 1.   +0.j, 1.   -0.j,
       1.   -0.j, 0.997-0.j, 1.   -0.j, 1.   +0.j, 1.   +0.j, 1.   +0.j,
       1.   -0.j, 1.   -0.j, 1.   -0.j, 0.998+0.j, 1.   -0.j, 1.   +0.j,
       0.999+0.j, 1.   +0.j, 1.   -0.j, 1.   +0.j, 1.   +0.j, 1.   +0.j,
       1.   -0.j, 1.   +0.j, 1.   -0.j, 1.   +0.j, 1.   -0.j, 1.   +0.j,
       1.   +0.j, 1.   -0.j, 1.   +0.j, 1.   +0.j, 1.   +0.j, 1.   -0.j,
       1.   +0.j, 0.999-0.j, 1.   -0.j, 1.   +0.j, 

#### Cut state results

In [120]:
np.array(
    [d['cut_state_fermion_parity'] for d in results]
)

array([-1.+5.75519038e-20j, -1.+9.85441574e-19j, -1.-1.29332586e-18j,
        1.-1.53332542e-18j, -1.-9.91975948e-19j,  1.+9.38465278e-19j,
       -1.-5.77921147e-19j,  1.+1.99990559e-18j, -1.+3.42295308e-20j,
        1.+4.27106195e-19j, -1.+2.44166406e-18j,  1.-1.41779133e-19j,
        1.+7.99597245e-19j, -1.-5.26011286e-19j,  1.+9.10481056e-19j,
        1.-1.18353665e-19j, -1.+1.03045152e-18j,  1.+1.18477273e-18j,
       -1.-1.34239666e-18j,  1.+6.57170816e-19j, -1.+2.13766735e-18j,
       -1.+1.82103200e-18j, -1.-2.03735967e-18j,  1.-2.48896824e-19j,
        1.-1.73041537e-18j,  1.-1.50030841e-19j, -1.-1.40205612e-18j,
        1.-4.81787621e-19j,  1.+3.66044086e-19j,  1.-2.05683425e-18j,
       -1.-4.93376030e-19j,  1.-4.01403810e-19j,  1.-1.08325778e-18j,
       -1.-1.00488681e-18j,  1.-3.83074600e-19j, -1.+8.95373163e-19j,
        1.-1.03332725e-19j, -1.-2.45345851e-19j, -1.+7.75557861e-20j,
       -1.+5.55880373e-19j,  1.+4.17854606e-20j,  1.+1.22597728e-18j,
       -1.+1.1240154

So the FP of the cut state can be even or odd. Interesting!

In [121]:
np.round(np.array(
    [d['cut_overlap'] for d in results]
), 3)

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1.

#### Defect op scores overlaps

In [122]:
np.round(np.array([
    [d['defect_ops_scores'][i][-1] for d in results]
    for i in range(3)
]), 5)

array([[0.99977, 0.99952, 0.99914, ..., 0.99998, 1.     , 0.99997],
       [1.     , 1.     , 1.     , ..., 1.     , 1.     , 1.     ],
       [0.99977, 0.99952, 0.99914, ..., 0.99998, 1.     , 0.99997]],
      shape=(3, 1000))

#### Phases

In [123]:
left_phases = np.array([
    d['left_defect_op_invariant'] for d in results
])

In [124]:
left_phases.shape

(1000,)

In [125]:
np.round(left_phases, 3)

array([-1.   -0.j, -1.   -0.j, -1.   -0.j, -1.   +0.j, -1.   -0.j,
       -1.   +0.j, -1.   +0.j, -1.   -0.j, -1.   -0.j, -1.   -0.j,
       -1.   -0.j, -1.   -0.j, -1.   +0.j, -1.   +0.j, -1.   -0.j,
       -1.   +0.j, -1.   +0.j, -1.   +0.j, -1.   +0.j, -1.   +0.j,
       -1.   -0.j, -1.   +0.j, -1.   -0.j, -1.   -0.j, -1.   -0.j,
       -1.   -0.j, -1.   -0.j, -1.   +0.j, -1.   +0.j, -1.   -0.j,
       -1.   -0.j, -1.   +0.j, -1.   -0.j, -1.   +0.j, -1.   +0.j,
       -1.   +0.j, -1.   -0.j, -1.   +0.j, -1.   +0.j, -1.   +0.j,
       -1.   +0.j, -1.   +0.j, -1.   -0.j, -1.   -0.j, -1.   -0.j,
       -1.   -0.j, -1.   -0.j, -1.   -0.j, -1.   +0.j, -1.   +0.j,
       -1.   -0.j, -1.   -0.j, -1.   +0.j, -1.   +0.j, -1.   -0.j,
       -1.   -0.j, -1.   -0.j, -1.   -0.j, -1.   +0.j, -1.   +0.j,
       -1.   -0.j, -1.   -0.j, -1.   +0.j, -1.   +0.j, -1.   +0.j,
       -1.   +0.j, -1.   -0.j, -1.   +0.j, -1.   +0.j, -1.   -0.j,
       -1.   -0.j, -1.   -0.j, -1.   -0.j, -1.   +0.j, -1.   +

In [126]:
right_phases = np.array([
    d['right_defect_op_invariant'] for d in results
])

In [127]:
right_phases.shape

(1000,)

In [128]:
np.round(right_phases, 3)

array([-1.   -0.j, -1.   +0.j, -1.   -0.j, -1.   -0.j, -1.   -0.j,
       -1.   -0.j, -1.   +0.j, -1.   -0.j, -1.   -0.j, -1.   -0.j,
       -0.999-0.j, -1.   -0.j, -1.   -0.j, -1.   -0.j, -1.   -0.j,
       -1.   -0.j, -1.   +0.j, -1.   -0.j, -1.   +0.j, -1.   -0.j,
       -1.   -0.j, -1.   +0.j, -1.   -0.j, -1.   +0.j, -1.   +0.j,
       -1.   -0.j, -1.   +0.j, -1.   +0.j, -1.   +0.j, -1.   -0.j,
       -1.   -0.j, -1.   +0.j, -1.   -0.j, -1.   +0.j, -1.   +0.j,
       -1.   +0.j, -1.   -0.j, -1.   -0.j, -1.   -0.j, -1.   +0.j,
       -1.   +0.j, -1.   +0.j, -1.   -0.j, -1.   -0.j, -1.   -0.j,
       -1.   +0.j, -1.   -0.j, -1.   +0.j, -1.   +0.j, -1.   +0.j,
       -1.   -0.j, -1.   +0.j, -1.   +0.j, -1.   +0.j, -1.   -0.j,
       -1.   -0.j, -1.   +0.j, -1.   -0.j, -1.   -0.j, -1.   +0.j,
       -1.   -0.j, -1.   -0.j, -1.   +0.j, -1.   +0.j, -1.   +0.j,
       -1.   +0.j, -1.   -0.j, -1.   -0.j, -1.   +0.j, -1.   -0.j,
       -1.   -0.j, -1.   -0.j, -1.   +0.j, -1.   +0.j, -1.   -

#### Cocyle information

In [129]:
np.array([
    d['cocycle_invariants'] for d in results
])

array([[-0.99999971+1.29302575e-08j, -0.99999923+2.16069929e-07j],
       [-0.99999592-9.43228721e-07j, -0.99999969-4.48196853e-08j],
       [-0.9999998 +3.68197828e-08j, -0.99999024+4.30178621e-07j],
       ...,
       [-0.99999947+1.24725497e-07j, -0.99987017-9.33844683e-06j],
       [-0.99999981+2.05620799e-09j, -0.9999952 -7.85456833e-07j],
       [-1.        +6.15333933e-11j, -0.99999338-3.26516321e-07j]],
      shape=(1000, 2))

In [130]:
cocycle_equation_array = np.array([d['cocycle_equation_output'] for d in results])
print(np.linalg.norm(cocycle_equation_array-1))
print(np.max(np.abs(cocycle_equation_array-1)))

0.039310883151499026
0.007815346029563464


Better!

In [131]:
bosonic_cocycle_equation_array = np.array([d['bosonic_cocycle_equation_output'] for d in results])

In [132]:
bosonic_cocycle_equation_array.shape

(1000, 2, 2, 2, 2)

In [133]:
np.linalg.norm(bosonic_cocycle_equation_array[..., -1, -1, -1] + 1)

np.float64(0.004047925864567223)

In [134]:
bosonic_cocycle_equation_array_copy = bosonic_cocycle_equation_array.copy()
bosonic_cocycle_equation_array_copy[..., -1, -1, -1] = 1
np.linalg.norm(bosonic_cocycle_equation_array_copy - 1)

np.float64(4.164446417378687e-15)

In [135]:
np.round(
    np.array([d['cocycles'][:, 0, 0] for d in results])
, 3)

array([[ 0.+1.j, -0.-1.j],
       [-0.+1.j,  0.-1.j],
       [ 0.+1.j, -0.-1.j],
       ...,
       [ 0.+1.j,  0.-1.j],
       [-0.-1.j, -0.+1.j],
       [-0.-1.j, -0.+1.j]], shape=(1000, 2))

In [136]:
np.round(
    np.array([d['fermion_decoration_from_cocycle'] for d in results])
, 3)

array([[-1.+0.j, -1.+0.j],
       [-1.-0.j, -1.+0.j],
       [-1.+0.j, -1.+0.j],
       ...,
       [-1.+0.j, -1.+0.j],
       [-1.-0.j, -1.-0.j],
       [-1.+0.j, -1.-0.j]], shape=(1000, 2))

# Product state - 2 site projectors and defect operators

In [137]:
domains_dict = {
    'num_system_sites': 32,
    'left_projector_sites': list(range(4, 8)),
    'right_projector_sites': list(range(22, 28)),
    'num_projector_pad_sites': 2,
    'num_defect_sites': 4,
    'fdlu_depth': 2,
    'fdlu_offset': 0
}

In [138]:
product_psi = get_product_qu_tensor_network(domains_dict['num_system_sites'])

In [139]:
results = list()

for _ in tqdm(range(1000)):
    current = find_invariants_via_projectors_from_random_state(
        product_psi,
        domains_dict,
        jw_even=True
    )
    results.append(current)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [14:28<00:00,  1.15it/s]


In [140]:
with open(r"results/z_4_tf_product_results.pkl", "wb") as file:
    pickle.dump(results, file)

### Analyze results

#### Projector scores

In [141]:
np.round(np.array([
    d['left_proj_vec_results'][0][-1]
    for d in results
]), 3)

array([ 0., -0.,  0., -0., -0.,  0., -0.,  0., -0.,  0., -0.,  0.,  0.,
       -0., -0.,  0.,  0.,  0., -0.,  0.,  0., -0., -0.,  0.,  0.,  0.,
        0., -0.,  0.,  0., -0.,  0.,  0.,  0.,  0., -0.,  0.,  0., -0.,
       -0.,  0., -0.,  0., -0.,  0.,  0., -0.,  0., -0., -0.,  0., -0.,
       -0.,  0.,  0., -0.,  0.,  0.,  0.,  0.,  0., -0., -0.,  0., -0.,
        0.,  0.,  0.,  0.,  0., -0., -0., -0.,  0.,  0.,  0.,  0., -0.,
        0.,  0.,  0., -0.,  0., -0.,  0.,  0.,  0., -0.,  0., -0.,  0.,
        0.,  0.,  0., -0., -0., -0., -0., -0.,  0., -0.,  0.,  0., -0.,
        0., -0., -0., -0., -0.,  0.,  0., -0.,  0.,  0.,  0.,  0., -0.,
       -0.,  0., -0.,  0., -0., -0.,  0., -0., -0., -0., -0., -0., -0.,
        0.,  0.,  0., -0., -0.,  0., -0., -0.,  0.,  0., -0., -0.,  0.,
       -0., -0., -0.,  0., -0., -0.,  0., -0., -0., -0., -0., -0.,  0.,
       -0., -0.,  0., -0.,  0.,  0.,  0.,  0., -0.,  0., -0., -0., -0.,
       -0., -0., -0.,  0.,  0.,  0., -0., -0., -0.,  0., -0.,  0

In [142]:
np.round(np.array([
    d['right_proj_vec_results'][0][-1]
    for d in results
]), 3)

array([-0.,  0., -0.,  0., -0.,  0.,  0.,  0., -0., -0.,  0., -0.,  0.,
       -0., -0.,  0., -0.,  0., -0., -0., -0., -0., -0.,  0.,  0., -0.,
        0.,  0.,  0., -0., -0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
        0.,  0., -0., -0., -0., -0., -0., -0.,  0., -0.,  0., -0.,  0.,
        0.,  0.,  0.,  0., -0., -0.,  0.,  0.,  0.,  0.,  0., -0., -0.,
        0.,  0.,  0.,  0., -0., -0.,  0., -0.,  0.,  0., -0., -0.,  0.,
       -0., -0., -0., -0., -0., -0.,  0.,  0.,  0.,  0., -0.,  0., -0.,
       -0.,  0.,  0., -0., -0., -0., -0., -0., -0.,  0., -0., -0., -0.,
       -0., -0.,  0., -0.,  0.,  0.,  0.,  0., -0., -0.,  0.,  0.,  0.,
        0., -0.,  0., -0., -0., -0., -0.,  0.,  0., -0., -0., -0.,  0.,
       -0., -0.,  0.,  0.,  0.,  0., -0., -0., -0.,  0., -0., -0.,  0.,
       -0.,  0.,  0., -0., -0.,  0., -0., -0., -0.,  0.,  0., -0.,  0.,
       -0.,  0., -0.,  0., -0.,  0., -0., -0.,  0.,  0., -0., -0., -0.,
        0.,  0., -0.,  0., -0.,  0.,  0.,  0.,  0.,  0.,  0., -0

In [143]:
def get_schmidt_vals_ratio(schmidt_vals):
    if len(schmidt_vals.data) > 1:
        return schmidt_vals.data[1]/schmidt_vals.data[0]
    else:
        return 0

In [144]:
np.round(np.array([
    get_schmidt_vals_ratio(d['left_proj_vec_results'][1])
    for d in results
]), 3)

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [145]:
np.round(np.array([
    get_schmidt_vals_ratio(d['right_proj_vec_results'][1])
    for d in results
]), 3)

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

#### Purities

In [146]:
np.round(np.array(
    [d['cut_rho_purity'] for d in results]
), 3)

array([1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j,
       1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j,
       1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j,
       1.-0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j,
       1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
       1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j,
       1.-0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j,
       1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j,
       1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j,
       1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j,
       1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j,
       1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j,
       1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j,
       1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j,
      

In [147]:
np.round(np.array(
    [d['sub_cut_rho_trace'] for d in results]
), 3)

array([1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j,
       1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j,
       1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j,
       1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j,
       1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
       1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j,
       1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j,
       1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j,
       1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j,
       1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j,
       1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j,
       1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j,
       1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j,
       1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j,
      

In [148]:
np.round(np.array(
    [d['sub_cut_rho_purity'] for d in results]
), 3)

array([1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j,
       1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j,
       1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j,
       1.-0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j,
       1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
       1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j,
       1.-0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j,
       1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j,
       1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j,
       1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j,
       1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j,
       1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j,
       1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j,
       1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j,
      

Purities are the same, which one could likely prove.

In [149]:
np.round(np.array(
    [d['sub_cut_rho_purity'] for d in results]
), 3)

array([1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j,
       1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j,
       1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j,
       1.-0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j,
       1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
       1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j,
       1.-0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j,
       1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j,
       1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j,
       1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j,
       1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j,
       1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j,
       1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j,
       1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j,
      

#### Cut state results

In [ ]:
np.array(
    [d['cut_state_fermion_parity'] for d in results]
)

So the FP of the cut state can be even or odd. Interesting!

In [ ]:
np.round(np.array(
    [d['cut_overlap'] for d in results]
), 3)

#### Defect op scores overlaps

In [ ]:
np.round(np.array([
    [d['defect_ops_scores'][i][-1] for d in results]
    for i in range(3)
]), 5)

#### Phases

In [ ]:
left_phases = np.array([
    d['left_defect_op_invariant'] for d in results
])

In [ ]:
left_phases.shape

In [ ]:
np.round(left_phases, 3)

In [ ]:
right_phases = np.array([
    d['right_defect_op_invariant'] for d in results
])

In [ ]:
right_phases.shape

In [ ]:
np.round(right_phases, 3)

#### Cocyle information

In [ ]:
np.array([
    d['cocycle_invariants'] for d in results
])

In [ ]:
cocycle_equation_array = np.array([d['cocycle_equation_output'] for d in results])
np.linalg.norm(cocycle_equation_array-1)

In [ ]:
bosonic_cocycle_equation_array = np.array([d['bosonic_cocycle_equation_output'] for d in results])

In [ ]:
bosonic_cocycle_equation_array.shape

In [ ]:
np.linalg.norm(bosonic_cocycle_equation_array - 1)

In [ ]:
np.round(
    np.array([d['cocycles'][:, 0, 0] for d in results])
, 3)

In [ ]:
np.round(
    np.array([d['fermion_decoration_from_cocycle'] for d in results])
, 3)

# Conclusions
Looks good!